# Escalation-Aware Support Workflow (LangGraph)

This notebook builds a support workflow that:
- routes easy issues to automated resolution
- routes medium issues to automated drafted response
- routes hard issues to human review and escalation notes


## Problem Framing

Support teams need fast responses for common requests and careful handling for risky tickets.
A single path for all tickets is inefficient. Conditional routing solves this by choosing the handling path dynamically.


## State Graph Overview

```text
START
  -> classify_ticket
       |
       | if difficulty == easy   -> auto_resolve
       | if difficulty == medium -> draft_response
       | if difficulty == hard   -> escalate_to_human

auto_resolve      -> END
draft_response    -> END
escalate_to_human -> END
```

The branch selection is implemented with `add_conditional_edges(...)` plus a router function.


## Step 1 - Install and Import

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph'])
print('langgraph installed')


langgraph installed


In [2]:
from typing import Literal, TypedDict
from langgraph.graph import END, START, StateGraph

print('Imports ready')


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports ready


## Step 2 - Define Workflow State

In [3]:
class SupportState(TypedDict):
    ticket_id: str
    customer_name: str
    ticket_text: str
    category: str
    difficulty: Literal['easy', 'medium', 'hard']
    sentiment: str
    routing_path: str
    response: str
    escalation_reason: str

print('SupportState defined')


SupportState defined


## Step 3 - Sample Tickets

In [4]:
tickets = [
    {
        'ticket_id': 'T-2001',
        'customer_name': 'Ava Martin',
        'ticket_text': 'How can I reset my password? I cannot log in.',
    },
    {
        'ticket_id': 'T-2002',
        'customer_name': 'Ravi Patel',
        'ticket_text': 'I was charged twice for my subscription this month. Please refund duplicate payment TXN-8881.',
    },
    {
        'ticket_id': 'T-2003',
        'customer_name': 'Emma Li',
        'ticket_text': 'URGENT: We suspect unauthorized access to our enterprise account and potential data exposure. Security team needed now.',
    },
    {
        'ticket_id': 'T-2004',
        'customer_name': 'Noah Brown',
        'ticket_text': 'What are your support hours and phone number?',
    },
    {
        'ticket_id': 'T-2005',
        'customer_name': 'Mia Garcia',
        'ticket_text': 'Our API has intermittent 502 errors affecting production traffic and SLA commitments.',
    },
]

print('Sample tickets:', len(tickets))


Sample tickets: 5


## Step 4 - Node 1: Classify Ticket (Heuristic)

In [5]:
def classify_ticket(state: SupportState) -> dict:
    text = state['ticket_text'].lower()

    if 'password' in text or 'support hours' in text or 'phone number' in text:
        category = 'account'
        difficulty = 'easy'
        sentiment = 'neutral'
    elif 'charged twice' in text or 'refund' in text or 'billing' in text:
        category = 'billing'
        difficulty = 'medium'
        sentiment = 'frustrated'
    elif 'unauthorized access' in text or 'data exposure' in text or 'security' in text:
        category = 'security'
        difficulty = 'hard'
        sentiment = 'angry'
    elif '502' in text or 'production' in text or 'sla' in text:
        category = 'technical'
        difficulty = 'hard'
        sentiment = 'frustrated'
    else:
        category = 'general'
        difficulty = 'medium'
        sentiment = 'neutral'

    print(f"classify_ticket: {state['ticket_id']} -> {difficulty}")
    return {'category': category, 'difficulty': difficulty, 'sentiment': sentiment}


## Step 5 - Node 2A: Easy Tickets -> Auto Resolve

In [6]:
def auto_resolve(state: SupportState) -> dict:
    if 'password' in state['ticket_text'].lower():
        response = (
            'Hi ' + state['customer_name'] + ',\n\n'
            'Use Forgot Password on the login page, enter your email, and follow the reset link. '
            'If the email does not arrive in 5 minutes, check spam and request a new link.\n\n'
            'Best, Support Team'
        )
    else:
        response = (
            'Hi ' + state['customer_name'] + ',\n\n'
            'Our support hours are Mon-Fri, 9:00 to 18:00 local time. '
            'Phone support is available on the account contact page.\n\n'
            'Best, Support Team'
        )

    return {'routing_path': 'auto_resolved', 'response': response, 'escalation_reason': ''}


## Step 6 - Node 2B: Medium Tickets -> Draft Automated Response

In [7]:
def draft_response(state: SupportState) -> dict:
    response = (
        'Hi ' + state['customer_name'] + ',\n\n'
        'Thanks for contacting us. We reviewed your billing concern and opened a finance check. '
        'If duplicate charge is confirmed, refund will be processed to your original payment method. '
        'We will send an update within one business day.\n\n'
        'Regards, Billing Support'
    )
    return {'routing_path': 'drafted_auto', 'response': response, 'escalation_reason': ''}


## Step 7 - Node 2C: Hard Tickets -> Human Escalation

In [8]:
def escalate_to_human(state: SupportState) -> dict:
    reason = (
        'Escalation required due to high-risk issue. '
        'Category=' + state['category'] + '; Sentiment=' + state['sentiment'] + '. '
        'Needs specialist triage and manual response approval.'
    )
    response = '[ESCALATED] Ticket sent to human specialist for immediate review.'
    return {'routing_path': 'escalated_human', 'response': response, 'escalation_reason': reason}


## Step 8 - Conditional Router

In [9]:
def route_by_difficulty(state: SupportState) -> str:
    difficulty = state.get('difficulty', 'medium')
    if difficulty == 'easy':
        return 'auto_resolve'
    if difficulty == 'hard':
        return 'escalate_to_human'
    return 'draft_response'

print('Router logic: easy -> auto_resolve, medium -> draft_response, hard -> escalate_to_human')


Router logic: easy -> auto_resolve, medium -> draft_response, hard -> escalate_to_human


## Conditional Routing Explanation

`add_conditional_edges('classify_ticket', route_by_difficulty, route_map)` means:
1. run `classify_ticket` first
2. call `route_by_difficulty(state)`
3. use returned key to select the next node from `route_map`

This is dynamic branching in LangGraph. The next step is data-dependent, not fixed.


## Step 9 - Build and Compile Graph

In [10]:
builder = StateGraph(SupportState)
builder.add_node('classify_ticket', classify_ticket)
builder.add_node('auto_resolve', auto_resolve)
builder.add_node('draft_response', draft_response)
builder.add_node('escalate_to_human', escalate_to_human)

builder.add_edge(START, 'classify_ticket')
builder.add_conditional_edges(
    'classify_ticket',
    route_by_difficulty,
    {
        'auto_resolve': 'auto_resolve',
        'draft_response': 'draft_response',
        'escalate_to_human': 'escalate_to_human',
    },
)
builder.add_edge('auto_resolve', END)
builder.add_edge('draft_response', END)
builder.add_edge('escalate_to_human', END)

app = builder.compile()
print('Graph compiled')


Graph compiled


## Step 10 - Run Workflow on All Tickets

In [11]:
results = []

for ticket in tickets:
    state = {
        'ticket_id': ticket['ticket_id'],
        'customer_name': ticket['customer_name'],
        'ticket_text': ticket['ticket_text'],
        'category': '',
        'difficulty': 'medium',
        'sentiment': '',
        'routing_path': '',
        'response': '',
        'escalation_reason': '',
    }

    result = app.invoke(state)
    results.append(result)

print('Processed tickets:', len(results))


classify_ticket: T-2001 -> easy
classify_ticket: T-2002 -> medium
classify_ticket: T-2003 -> hard
classify_ticket: T-2004 -> easy
classify_ticket: T-2005 -> hard
Processed tickets: 5


## Step 11 - Show Per-Ticket Routing Decisions

In [12]:
for r in results:
    print('=' * 70)
    print('Ticket:', r['ticket_id'], '| Customer:', r['customer_name'])
    print('Category:', r['category'], '| Difficulty:', r['difficulty'], '| Path:', r['routing_path'])
    print('Response preview:', r['response'][:140])
    if r['routing_path'] == 'escalated_human':
        print('Escalation reason:', r['escalation_reason'])


Ticket: T-2001 | Customer: Ava Martin
Category: account | Difficulty: easy | Path: auto_resolved
Response preview: Hi Ava Martin,

Use Forgot Password on the login page, enter your email, and follow the reset link. If the email does not arrive in 5 minute
Ticket: T-2002 | Customer: Ravi Patel
Category: billing | Difficulty: medium | Path: drafted_auto
Response preview: Hi Ravi Patel,

Thanks for contacting us. We reviewed your billing concern and opened a finance check. If duplicate charge is confirmed, ref
Ticket: T-2003 | Customer: Emma Li
Category: security | Difficulty: hard | Path: escalated_human
Response preview: [ESCALATED] Ticket sent to human specialist for immediate review.
Escalation reason: Escalation required due to high-risk issue. Category=security; Sentiment=angry. Needs specialist triage and manual response approval.
Ticket: T-2004 | Customer: Noah Brown
Category: account | Difficulty: easy | Path: auto_resolved
Response preview: Hi Noah Brown,

Our support hours are 

## Step 12 - Routing Summary Dashboard

In [13]:
summary = {'auto_resolved': 0, 'drafted_auto': 0, 'escalated_human': 0}

for r in results:
    summary[r['routing_path']] = summary.get(r['routing_path'], 0) + 1

print('Routing summary:')
for k in ['auto_resolved', 'drafted_auto', 'escalated_human']:
    print('-', k + ':', summary.get(k, 0))


Routing summary:
- auto_resolved: 2
- drafted_auto: 1
- escalated_human: 2


## Step 13 - Simple Validation Checks

In [14]:
easy_paths_ok = all(r['routing_path'] == 'auto_resolved' for r in results if r['difficulty'] == 'easy')
hard_paths_ok = all(r['routing_path'] == 'escalated_human' for r in results if r['difficulty'] == 'hard')

print('Easy tickets auto-resolved:', easy_paths_ok)
print('Hard tickets escalated to human:', hard_paths_ok)

if not easy_paths_ok or not hard_paths_ok:
    raise RuntimeError('Routing validation failed')

print('Conditional routing behavior validated')


Easy tickets auto-resolved: True
Hard tickets escalated to human: True
Conditional routing behavior validated


## Key Takeaways

1. Conditional routing in LangGraph is implemented with a router function and `add_conditional_edges`.
2. Easy issues can be closed automatically to reduce queue load.
3. Hard issues are escalated to humans for safety and accountability.
4. Branch-specific nodes keep each resolution strategy focused and auditable.
